# MvDEC seed-42 pretrain/final alignment audit
Evaluation-only audit. It never trains or modifies the canonical artifacts.

In [ ]:
import json, pickle, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.linalg import orthogonal_procrustes
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

print("TensorFlow", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "loguru>=0.7.3"])
tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
search_roots = (INPUT_ROOT, WORKING_ROOT)
artifact_paths = sorted(
    path
    for root in search_roots
    for path in root.rglob("artifact.pkl")
    if "mvdec_2025_paper_informed_full_batch_v2" in str(path)
)
assert len(artifact_paths) == 3, artifact_paths
source_files = sorted(
    path
    for root in search_roots
    for path in root.rglob("src/representation_learning/MVDEC_dense.py")
)
assert len(source_files) == 1, source_files
repo = source_files[0].parents[2]
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo / "src" / "representation_learning"))
print("repo", repo)
print("artifacts", artifact_paths)


In [ ]:
import representation_learning.MVDEC_dense as mvdec
from sklearn.datasets import fetch_20newsgroups, fetch_rcv1

# Populate sklearn cache explicitly; project loader is intentionally fail-closed.
fetch_20newsgroups(subset="all", download_if_missing=True)
fetch_rcv1(download_if_missing=True)
from pipeline.public_data import (
    TWENTY_NEWS_RELEASE_NPZ,
    TWENTY_NEWS_SKLEARN_SOURCE,
    load_dekm_public_dataset,
)


def acc(y, predicted, n_clusters):
    matrix = np.zeros((n_clusters, n_clusters), dtype=np.int64)
    np.add.at(matrix, (predicted.astype(int), y.astype(int)), 1)
    rows, columns = linear_sum_assignment(-matrix)
    return float(matrix[rows, columns].sum() / len(y))


def evaluate(embedding, y, n_clusters, seed, n_init):
    predicted = KMeans(
        n_clusters=n_clusters,
        n_init=n_init,
        random_state=seed,
    ).fit_predict(embedding)
    return (
        predicted,
        acc(y, predicted, n_clusters),
        float(normalized_mutual_info_score(y, predicted)),
    )


def align_stats(view1, view2):
    norm1 = np.linalg.norm(view1, axis=1)
    norm2 = np.linalg.norm(view2, axis=1)
    denominator = norm1 * norm2
    cosine = np.divide(
        np.sum(view1 * view2, axis=1),
        denominator,
        out=np.zeros(len(view1)),
        where=denominator > 0,
    )
    retained = np.divide(
        np.linalg.norm((view1 + view2) / 2, axis=1),
        (norm1 + norm2) / 2,
        out=np.zeros(len(view1)),
        where=(norm1 + norm2) > 0,
    )
    centered1 = view1 - view1.mean(axis=0)
    centered2 = view2 - view2.mean(axis=0)
    correlation = np.corrcoef(centered1.T, centered2.T)[
        : view1.shape[1], view1.shape[1] :
    ]
    rotation, _ = orthogonal_procrustes(centered2, centered1)
    aligned = (view1 + (centered2 @ rotation + view1.mean(axis=0))) / 2
    return {
        "paired_cosine_mean": float(cosine.mean()),
        "same_dim_corr_mean": float(np.diag(correlation).mean()),
        "retained_norm_mean": float(retained.mean()),
    }, aligned


In [ ]:
out = Path("/kaggle/working/mvdec-seed42-pretrain-final-audit")
out.mkdir(parents=True, exist_ok=True)
records = []
alignments = []
replay_checks = []

for artifact_path in artifact_paths:
    with artifact_path.open("rb") as file:
        artifact = pickle.load(file)

    dataset_name = artifact["dataset"]
    run_dir = artifact_path.parent
    n_clusters = int(artifact["n_clusters"])
    seed = int(artifact["config"]["seed"])
    n_init = int(artifact["kmeans_n_init"])
    mvdec.ds_name = dataset_name
    mvdec.input_shape = int(artifact["input_dim"])
    mvdec.n_clusters = n_clusters
    mvdec.hidden_units = int(artifact["view1_latent_dim"])

    twenty_news_source = (
        TWENTY_NEWS_SKLEARN_SOURCE
        if dataset_name == "20NEWS"
        else TWENTY_NEWS_RELEASE_NPZ
    )
    dataset = load_dekm_public_dataset(
        dataset_name,
        repo / "external_repos/DEKM/datasets",
        normalization=artifact["protocol_contract"]["preprocessing"][
            "input_normalization"
        ],
        twenty_news_source=twenty_news_source,
    )
    assert dataset.source_id == artifact["source_id"]
    assert dataset.source_sha256 == artifact["source_sha256"]
    x = dataset.X
    y = dataset.y

    stages = (
        (
            "pretrain",
            run_dir / "pretrain_view1.weights.h5",
            run_dir / "pretrain_view2.weights.h5",
        ),
        (
            "final",
            run_dir / "final_view1.weights.h5",
            run_dir / "final_view2.weights.h5",
        ),
    )
    for stage, view1_path, view2_path in stages:
        assert view1_path.is_file() and view2_path.is_file()
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        model1 = mvdec.model_view1(weights_path=view1_path)
        model2 = mvdec.model_view2(
            weights_path=view2_path,
            architecture_id=artifact["view2_architecture_id"],
        )
        output1 = model1.predict(x, batch_size=512, verbose=0)
        output2 = model2.predict(x, batch_size=512, verbose=0)
        view1 = output1[:, : artifact["view1_latent_dim"]]
        view2 = output2[:, : artifact["view2_latent_dim"]]
        fused = (view1 + view2) / 2

        if stage == "final":
            mismatches = {
                "dataset": dataset_name,
                "view1_max_abs_error": float(
                    np.max(np.abs(view1 - artifact["h_view1"]))
                ),
                "view2_max_abs_error": float(
                    np.max(np.abs(view2 - artifact["h_view2"]))
                ),
                "fused_max_abs_error": float(
                    np.max(np.abs(fused - artifact["h_fused"]))
                ),
            }
            replay_checks.append(mismatches)
            assert mismatches["view1_max_abs_error"] < 1e-5, mismatches
            assert mismatches["view2_max_abs_error"] < 1e-5, mismatches
            assert mismatches["fused_max_abs_error"] < 1e-5, mismatches

        stats, aligned = align_stats(view1, view2)
        alignments.append({"dataset": dataset_name, "stage": stage, **stats})
        representations = (
            ("view1", view1),
            ("view2", view2),
            ("fused", fused),
            ("procrustes_control", aligned),
            ("concat_control", np.concatenate([view1, view2], axis=1)),
        )
        for name, embedding in representations:
            labels, accuracy, nmi = evaluate(
                embedding,
                y,
                n_clusters,
                seed,
                n_init,
            )
            records.append(
                {
                    "dataset": dataset_name,
                    "stage": stage,
                    "representation": name,
                    "acc": accuracy,
                    "nmi": nmi,
                    "seed": seed,
                    "kmeans_n_init": n_init,
                }
            )
            pd.DataFrame(
                {
                    "sample_index": np.arange(len(labels)),
                    "cluster": labels,
                    "true_label": y,
                }
            ).to_csv(
                out / f"{dataset_name.lower()}_{stage}_{name}_assignments.csv",
                index=False,
            )

        del model1, model2, output1, output2, view1, view2, fused
        tf.keras.backend.clear_session()

metrics_frame = pd.DataFrame(records)
alignment_frame = pd.DataFrame(alignments)
replay_frame = pd.DataFrame(replay_checks)
metrics_frame.to_csv(out / "pretrain_final_metrics.csv", index=False)
alignment_frame.to_csv(out / "pretrain_final_alignment.csv", index=False)
replay_frame.to_csv(out / "final_weight_replay_checks.csv", index=False)
display(metrics_frame)
display(alignment_frame)
display(replay_frame)


In [ ]:
manifest = {
    "audit_type": "evaluation_only_pretrain_vs_final",
    "training_commit": "9b0c89afe522c009354bdd783b70fd182ec95d22",
    "source_kernel": "vandatzz/mvdec-2025-v2-three-datasets-seed-42",
    "files": sorted(path.name for path in out.iterdir()),
}
(out / "audit_manifest.json").write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(manifest, indent=2))
